# VISTA Challenge — Google Colab Runner

**Pipeline**: YOLO tracker + Qwen-VL captioner + severity detection  
**GPU**: T4 (16 GB) via Colab  
**Output**: `predictions_tracks.csv`, `predictions_mot.csv`, annotated video

### Steps
1. Upload `VISTA.zip` to your Google Drive
2. Mount Drive and unzip the project
3. Install dependencies
4. Upload your video
5. (Optional) Fine-tune YOLO on VistaCrash
6. Run the pipeline
7. Measure FPS
8. Evaluate with BERTScore

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

## 0 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1 — Mount Google Drive & unzip project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# ── EDIT: path to your zip file on Drive ──────────────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/VISTA.zip'
PROJECT_DIR = '/content/VISTA'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(PROJECT_DIR):
    !unzip -q "{ZIP_PATH}" -d /content/
    print(f'Unzipped to {PROJECT_DIR}')
else:
    print(f'{PROJECT_DIR} already exists, skipping unzip')

os.chdir(PROJECT_DIR)
!pwd
!ls

## 2 — Install dependencies

> This takes ~5-10 min on first run. Colab resets on reconnect — re-run if needed.

In [ ]:
# Core dependencies for the T4 pipeline
# We skip vllm (requires A100+) and use HuggingFace + unsloth 4-bit instead
!pip install -q \
    torch torchvision --index-url https://download.pytorch.org/whl/cu121 \
    2>/dev/null || true

!pip install -q \
    ultralytics \
    transformers>=4.57.1 \
    qwen-vl-utils>=0.0.14 \
    json-repair \
    opencv-python-headless \
    Pillow \
    bert-score \
    accelerate \
    bitsandbytes

# Unsloth for 4-bit Qwen3-VL on T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print('\n✅ Dependencies installed')

In [ ]:
# Verify key imports
import torch, cv2, ultralytics, transformers
from PIL import Image
print(f'torch       {torch.__version__}')
print(f'ultralytics {ultralytics.__version__}')
print(f'transformers {transformers.__version__}')
print(f'CUDA        {torch.cuda.is_available()}')

## 3 — HuggingFace cache & model choice

The T4 has **16 GB VRAM**. Model recommendations:

| Model | VRAM | Notes |
|---|---|---|
| `Qwen/Qwen2.5-VL-7B-Instruct` | ~14 GB fp16 | fits on T4, best quality |
| `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` | ~5 GB | ✅ recommended for T4 |
| `Qwen/Qwen3-VL-8B-Instruct` | ~15 GB fp16 | may OOM on T4 |
| `unsloth/Qwen3-VL-8B-Instruct-bnb-4bit` | ~6 GB | ✅ fits comfortably |

**Set `MODEL_ID` below.**

In [ ]:
import os

# ── EDIT these paths ──────────────────────────────────────────────────────────
MODEL_ID   = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'  # change if needed
VIDEO_PATH = 'data/sequences/DroneSim.mp4'               # path to your input video
YOLO_MODEL = 'yolo12x.pt'                                # base YOLO or your fine-tuned .pt
OUT_DIR    = './out/colab_run'
# ─────────────────────────────────────────────────────────────────────────────

os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Model   : {MODEL_ID}')
print(f'Video   : {VIDEO_PATH}')
print(f'YOLO    : {YOLO_MODEL}')
print(f'Output  : {OUT_DIR}')

## 4 — Upload your video (if not already on Drive)

In [ ]:
import os
if not os.path.exists(VIDEO_PATH):
    from google.colab import files
    print('Video not found — upload it now:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.makedirs('data/sequences', exist_ok=True)
    os.rename(fname, VIDEO_PATH)
    print(f'Saved to {VIDEO_PATH}')
else:
    print(f'Video found: {VIDEO_PATH}')

## 5 — (Optional) Fine-tune YOLO on VistaCrash

Skip this cell if you already have a fine-tuned checkpoint.  
Fine-tuning gives the `crashed_car` class and improves severity recall.

In [ ]:
RUN_FINETUNE = False  # Set to True to fine-tune

if RUN_FINETUNE:
    from ultralytics import YOLO
    yolo_ft = YOLO('yolo12x.pt')
    yolo_ft.train(
        data='data/VistaCrash/data.yaml',
        epochs=50,
        imgsz=640,
        batch=8,          # T4-safe batch size
        device=0,
        project='runs/detect',
        name='vistacrashe_x',
        exist_ok=True,
    )
    YOLO_MODEL = 'runs/detect/vistacrashe_x/weights/best.pt'
    print(f'Fine-tuned checkpoint: {YOLO_MODEL}')
else:
    print('Skipping fine-tuning — using', YOLO_MODEL)

## 6 — Write config for this run

In [ ]:
import yaml, textwrap

# Adjust stride to stay >=5 FPS on the shallow stage:
# Qwen runs every CAPTION_STRIDE frames; YOLO runs every frame.
CAPTION_STRIDE = 30   # lower = more captions, slower total FPS

cfg = {
    'input':  {'video': VIDEO_PATH},
    'output': {'dir': OUT_DIR},
    'device': 'cuda:0',
    'save_annotated_frames_frequency': 30,
    'merge_method': 'update',
    'yolo': {
        'model': YOLO_MODEL,
        'base_model': 'yolo12x.pt',
        'iou_match_threshold': 0.3,
        'conf': 0.25,
    },
    'qwen': {
        'model_id': MODEL_ID,
        'every_n_frames': CAPTION_STRIDE,
        'num_frames': 1,
        'max_new_tokens': 512,
        'image_target_size': 1024,
        'system_prompt': textwrap.dedent("""\
            You are a computer vision assistant analysing drone footage of a road
            accident scene. Detect every relevant object and assign it EXACTLY ONE
            label chosen from the two lists below. You MUST use the exact string as
            written — do not paraphrase, abbreviate, or invent new labels.

            Vehicle labels (pick the one that best describes the vehicle's damage):
              "undamaged"
              "minor damage"
              "heavily damaged"
              "overturned"
              "on fire"
              "ambulance on scene"
              "fire truck on scene"
              "police on scene"
              "emergency vehicle"

            Person labels (pick the one that best describes the person's status):
              "standing"
              "running"
              "helping"
              "calling for help"
              "injured, sitting"
              "injured, lying"
              "unconscious"

            Output format — return a valid JSON array ONLY, no other text:
            [{"bbox_2d": [xmin, ymin, xmax, ymax], "label": "<exact label from list>"}, ...]

            Example:
            [{"bbox_2d": [10, 30, 200, 160], "label": "heavily damaged"}, {"bbox_2d": [220, 50, 260, 120], "label": "injured, lying"}]"""),
        'user_prompt': '',
    },
}

CONFIG_PATH = '/content/VISTA/config_colab.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print(f'Config written to {CONFIG_PATH}')
print(yaml.dump(cfg))

## 7 — Run the pipeline

This runs the full YOLO + Qwen-VL + severity pipeline.  
FPS is printed at the end.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'qwen_yolo.py', '--config', CONFIG_PATH],
    cwd='/content/VISTA',
    capture_output=False,   # stream stdout live
)
print('\nExit code:', result.returncode)

## 8 — Measure FPS (shallow stage only, no Qwen)

This measures **only YOLO tracking** — the "shallow stage" the challenge requires ≥ 5 FPS on.

In [ ]:
import time, cv2, torch
from ultralytics import YOLO

yolo_bench = YOLO(YOLO_MODEL)

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Cannot open {VIDEO_PATH}'

# Warm-up: 10 frames
for _ in range(10):
    ret, frame = cap.read()
    if not ret: break
    yolo_bench.track(frame, persist=False, verbose=False, conf=0.05)

# Benchmark: 100 frames
N = 100
t0 = time.perf_counter()
for i in range(N):
    ret, frame = cap.read()
    if not ret: break
    yolo_bench.track(frame, persist=True, verbose=False, conf=0.05)
elapsed = time.perf_counter() - t0
cap.release()

fps = i / elapsed
status = '✅ PASS' if fps >= 5 else '❌ FAIL'
print(f'Shallow stage FPS over {i} frames: {fps:.2f}  {status} (threshold: 5 FPS)')

## 9 — Evaluate with BERTScore-F1

BERTScore-F1 is the **primary ranking metric**.  
You need a ground-truth `predictions_tracks.csv` with a `caption` column to compare against.

Format of both files:  
`video_id, track_id, frame_start, frame_end, caption`

In [ ]:
import pandas as pd
from bert_score import score as bert_score

PRED_CSV = f'{OUT_DIR}/predictions_tracks.csv'
GT_CSV   = '/content/drive/MyDrive/gt_tracks.csv'  # ← your ground-truth file

pred_df = pd.read_csv(PRED_CSV)
print(f'Predictions: {len(pred_df)} tracks')
print(pred_df.head())

if not __import__('os').path.exists(GT_CSV):
    print('\n⚠️  No ground-truth CSV found at', GT_CSV)
    print('Set GT_CSV to your ground-truth file path and re-run this cell.')
else:
    gt_df = pd.read_csv(GT_CSV)

    # Match on (video_id, track_id)
    merged = pred_df.merge(gt_df, on=['video_id', 'track_id'], suffixes=('_pred', '_gt'))
    if merged.empty:
        print('⚠️  No matching tracks between predictions and ground truth.')
    else:
        candidates = merged['caption_pred'].fillna('').tolist()
        references = merged['caption_gt'].fillna('').tolist()

        P, R, F1 = bert_score(
            candidates, references,
            lang='en',
            model_type='distilbert-base-uncased',  # lighter model for T4
            verbose=True,
        )
        print(f'\nBERTScore  P={P.mean():.4f}  R={R.mean():.4f}  F1={F1.mean():.4f}')
        print(f'BERTScore-F1 (primary metric): {F1.mean():.4f}')

## 10 — Inspect outputs

In [ ]:
import pandas as pd, os

tracks_csv = f'{OUT_DIR}/predictions_tracks.csv'
mot_csv    = f'{OUT_DIR}/predictions_mot.csv'

if os.path.exists(tracks_csv):
    df = pd.read_csv(tracks_csv)
    print(f'--- predictions_tracks.csv ({len(df)} rows) ---')
    print(df.to_string(index=False))
else:
    print('predictions_tracks.csv not found — did the pipeline run?')

if os.path.exists(mot_csv):
    df2 = pd.read_csv(mot_csv)
    print(f'\n--- predictions_mot.csv ({len(df2)} rows, first 10) ---')
    print(df2.head(10).to_string(index=False))

In [ ]:
# Preview the annotated video (first 5 seconds)
import cv2
from IPython.display import HTML
from base64 import b64encode
import os

video_out = f'{OUT_DIR}/annotated.mp4'
if os.path.exists(video_out):
    with open(video_out, 'rb') as f:
        data = f.read()
    b64 = b64encode(data).decode()
    display(HTML(f"""
    <video width="800" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    """))
else:
    print(f'Video not found at {video_out}')

## 11 — Download outputs to your machine

In [ ]:
from google.colab import files
import os

for fname in [f'{OUT_DIR}/predictions_tracks.csv', f'{OUT_DIR}/predictions_mot.csv']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'Downloading {fname}')
    else:
        print(f'Not found: {fname}')

# Optional: copy results back to Drive
# import shutil
# shutil.copytree(OUT_DIR, '/content/drive/MyDrive/VISTA_results', dirs_exist_ok=True)

---
## Notes & Tips for T4

| | |
|---|---|
| **OOM on Qwen?** | Use `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` and lower `image_target_size` to `512` |
| **FPS too low?** | Increase `every_n_frames` (e.g. 60) — YOLO always runs every frame, Qwen only every N frames |
| **Challenge threshold** | ≥ 5 FPS on the **shallow stage** (YOLO only) — measured in Cell 8 |
| **Primary metric** | BERTScore-F1 on `caption` column — measured in Cell 9 |
| **Tie-breakers** | MOTA and IDF1 from `predictions_mot.csv` (use `py-motmetrics`) |
| **Colab disconnects** | Save results to Drive at the end of Cell 11 |
| **No video?** | Use `data/sequences/DroneSim.mp4` already in the project, or upload yours in Cell 4 |